# Config

## Environment

### External packages

In [ ]:

# Interactive figures (ipympl): pan/zoom/rotate the matplotlib figures inline. Requires `ipympl`.
%matplotlib widget
# Re-import the src/results modules on every edit, so changing a .py takes effect without a restart.
%load_ext autoreload
%autoreload 2

import os
import sys
import json
import numpy as np
import pandas as pd
import ipywidgets as widgets
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

### Routing

In [ ]:
# Resolve repo paths and put src/ + the results code dirs on sys.path, exactly as each script does
# when run from the CLI. Run this notebook from the repo root.
ROOT = Path.cwd()
RAW = ROOT / "data" / "options" / "raw"
SRC = ROOT / "src"
RESULTS_CODE = SRC / "results"
for _p in (str(ROOT), str(SRC), str(RESULTS_CODE),
           str(RESULTS_CODE / "smiles"), str(RESULTS_CODE / "surfaces"),
           str(RESULTS_CODE / "tables")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

### Custom modules

In [ ]:
# Each results module exposes main(model=None, objective=None, save=..., show=...). With save=False
# nothing is written to disk; with show=True the plot modules render their figures inline (ipympl).
# Passing model/objective overrides the defaults in src/results/_results_config.py for that call only.
import _verify_completeness
import validate_calibrations
import wing_residuals
import smiles
import plot_surfaces
from plot_surfaces import price_grid, smile_for
from config import calib_paths, spec_path

# --- untracked modules
import plotters

## Model selection

In [ ]:
MODEL, OBJECTIVE = "bates", "vol"

In [ ]:
SPEC_FILE = spec_path(model=MODEL, objective=OBJECTIVE)
saved = json.loads(SPEC_FILE.read_text())
saved.pop('_run')
PARAM_ORDER = saved["PARAM_ORDER" if MODEL == "heston" else "BATES_PARAM_ORDER"]

# Data inspection

## Completeness

In [ ]:
# Completeness audit: do calibrations.csv + rejections.csv partition every raw trading day exactly
# once? Prints the report; returns 0 if clean, 1 if any discrepancy.
_verify_completeness.main(model=MODEL, objective=OBJECTIVE)

## Validation

In [ ]:
# Grade the run (fit quality, economic-reasonability flags, cross-day stability). save=False keeps it
# read-only (no validation.csv written); the returned frame is the per-day graded table.
val = validate_calibrations.main(model=MODEL, objective=OBJECTIVE, save=False)
val

## Wing residuals

In [ ]:
# Residual (model_iv - market_iv) stratified by moneyness / maturity. Prints the three tables and
# returns (per-contract frame, dict of bucket frames). Read-only; no save flag.
resid_df, resid_tables = wing_residuals.main(model=MODEL, objective=OBJECTIVE)
resid_tables["abs"]

# Plotting

## Date selection

In [ ]:
# Pick a day to inspect (here: the best-fit day of the chosen run).
_cal = pd.read_csv(calib_paths(MODEL, OBJECTIVE)[0]).sort_values("iv_rmse").reset_index(drop=True)
target_date = _cal["date"][0]
best_dates = _cal["date"][:15]
worst_dates = _cal["date"][-15:]

## Series

In [ ]:
dt_cal = _cal.copy()
dt_cal['date'] = pd.to_datetime(dt_cal['date'])
dt_cal = dt_cal.set_index('date').sort_index(ascending=True)
dt_cal

In [ ]:
plotters.PlotCols(dt_cal, col_names=['spot_price']+PARAM_ORDER+['spot_price', 'risk_free_rate', 'dividend_rate', 'iv_rmse', 'rmse', 'feller'])

## Smiles & surfaces (interactive, not saved)

In [ ]:
# Interactive mode OFF so the figures are not auto-shown as they are built (that auto-show, under
# ipympl/VS Code, also emits a duplicate static PNG next to the widget). We render them ourselves
# below and leave interactive mode off -- ipympl canvases stay pan/zoom/rotate-able regardless.
plt.ioff()

### Surfaces

In [ ]:
# save=False writes nothing to results/; show=True just leaves the figures open for us to render.
surf_figs, day_results = plot_surfaces.main(
    model=MODEL, objective=OBJECTIVE, target_date=target_date, save=False, show=True)

# Render every figure once, as a SINGLE stacked widget tree. Pushing several canvases via separate
# display() calls in one cell can drop one under ipympl (the dropped figure renders blank/nothing);
# collecting them into one VBox renders all of them. Using fig.canvas (not the Figure) emits only the
# interactive widget, never a static PNG.
_canvases = [f.canvas for f in [*surf_figs.values()] if f is not None]
widgets.VBox(_canvases)

### Best fit smiles

### Worst fit smiles

### Per-maturity smiles (each maturity on its own axes)

In [ ]:
# Plot each calibrated maturity of `target_date` on its OWN axes, reusing smiles._save_day_figure.
# _save_day_figure draws every maturity it finds in the scatter that _load_test_scatter returns, on a
# shared put/call pair. To split them, we temporarily patch smiles._load_test_scatter to hand back a
# single maturity's contracts per call, so each call yields a one-maturity figure (its own put/call
# axes). save=False keeps it read-only; show=True leaves the figures open for us to render.
from smiles import _save_day_figure
from _results_utils import load_calibrations_by_date, _load_test_scatter, _day_from_row

_cal_file, _, _tests_dir = calib_paths(MODEL, OBJECTIVE)
_row = load_calibrations_by_date(_cal_file).asof(pd.Timestamp(target_date))
_day = _day_from_row(_row, model=MODEL)

_full = _load_test_scatter(_day['tag'], tests_dir=_tests_dir)
_mats = sorted(_full['days_to_maturity'].unique()) if _full is not None else []

mat_figs = {}
_orig_loader = smiles._load_test_scatter
try:
    for _t in _mats:
        # Bind _t per iteration so the closure captures this maturity, not the loop variable.
        smiles._load_test_scatter = (
            lambda tag, tests_dir=None, _t=_t: _full[_full['days_to_maturity'] == _t].copy())
        _fig = _save_day_figure(_day, use_legend=True, model=MODEL,
                                tests_dir=_tests_dir, save=False, show=True)
        if _fig is not None:           # None => maturity fell outside [TMIN, TMAX] and was skipped
            mat_figs[_t] = _fig
finally:
    smiles._load_test_scatter = _orig_loader

# One stacked widget tree (see the surfaces cell for why a single VBox, not many display() calls).
widgets.VBox([f.canvas for f in mat_figs.values()])